In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report

sf = pd.read_csv("data/processed/humaid_safety.csv")

HOLDOUT = ["maryland_floods_2018", "srilanka_floods_2017"]   # events the model never sees
seen = sf[~sf["event"].isin(HOLDOUT)]
train = seen[seen["split"] == "train"]
test_in = seen[seen["split"] == "test"]
test_out = sf[sf["event"].isin(HOLDOUT)]
print(len(train), "train |", len(test_in), "test (same events) |", len(test_out), "test (unseen events)")

16986 train | 4808 test (same events) | 927 test (unseen events)


In [2]:
dummy = DummyClassifier(strategy="most_frequent").fit(train["text"], train["safety_label"])
print("NAIVE (same events):")
print(classification_report(test_in["safety_label"], dummy.predict(test_in["text"]), zero_division=0))

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, lowercase=True)
clf = LogisticRegression(max_iter=1000, class_weight="balanced").fit(vec.fit_transform(train["text"]), train["safety_label"])

for name, data in [("SAME EVENTS", test_in), ("UNSEEN EVENTS (floods)", test_out)]:
    print(f"\nTF-IDF + LogReg, {name}:")
    print(classification_report(data["safety_label"], clf.predict(vec.transform(data["text"]))))

NAIVE (same events):
              precision    recall  f1-score   support

      hazard       0.00      0.00      0.00      1001
  non_hazard       0.79      1.00      0.88      3807

    accuracy                           0.79      4808
   macro avg       0.40      0.50      0.44      4808
weighted avg       0.63      0.79      0.70      4808


TF-IDF + LogReg, SAME EVENTS:
              precision    recall  f1-score   support

      hazard       0.83      0.92      0.87      1001
  non_hazard       0.98      0.95      0.96      3807

    accuracy                           0.94      4808
   macro avg       0.90      0.94      0.92      4808
weighted avg       0.95      0.94      0.95      4808


TF-IDF + LogReg, UNSEEN EVENTS (floods):
              precision    recall  f1-score   support

      hazard       0.69      0.60      0.64        96
  non_hazard       0.95      0.97      0.96       831

    accuracy                           0.93       927
   macro avg       0.82      0.79 

In [3]:
tests = [
    "water is rushing down 5th avenue, cars are stuck up to the doors",
    "huge tree fell across the road on Queens Blvd, nobody can pass",
    "my neighbor is playing loud music again",
    "basement flooded with sewage after the storm",
    "the street is a river near the subway entrance",
    "nice weather today, going for a walk",
    "big branch came down on Atlantic Ave and blocks the lane",
]
probs = clf.predict_proba(vec.transform(tests))
hz = list(clf.classes_).index("hazard")
for t, p in zip(tests, probs):
    print(f"hazard prob {p[hz]:.2f}  {t}")

hazard prob 0.20  water is rushing down 5th avenue, cars are stuck up to the doors
hazard prob 0.48  huge tree fell across the road on Queens Blvd, nobody can pass
hazard prob 0.21  my neighbor is playing loud music again
hazard prob 0.41  basement flooded with sewage after the storm
hazard prob 0.49  the street is a river near the subway entrance
hazard prob 0.11  nice weather today, going for a walk
hazard prob 0.29  big branch came down on Atlantic Ave and blocks the lane


In [4]:
n311 = pd.read_csv("data/processed/nyc311_ida_labeled.csv").dropna(subset=["text", "label"])
n311 = n311.sample(10000, random_state=0)
n311["y"] = (n311["label"] != "non_hazard").map({True: "hazard", False: "non_hazard"})
print("Tweet-trained model on NYC 311 reports:")
print(classification_report(n311["y"], clf.predict(vec.transform(n311["text"]))))

Tweet-trained model on NYC 311 reports:
              precision    recall  f1-score   support

      hazard       0.42      0.17      0.24      1032
  non_hazard       0.91      0.97      0.94      8968

    accuracy                           0.89     10000
   macro avg       0.66      0.57      0.59     10000
weighted avg       0.86      0.89      0.87     10000



In [5]:
import numpy as np

dev = seen[seen["split"] == "dev"]
hz = list(clf.classes_).index("hazard")
p_dev = clf.predict_proba(vec.transform(dev["text"]))[:, hz]
y_dev = (dev["safety_label"] == "hazard").astype(int).values

rows = []
for th in np.arange(0.20, 0.71, 0.05):
    pred = (p_dev >= th).astype(int)
    tp = int(((pred == 1) & (y_dev == 1)).sum())
    prec = tp / max(int(pred.sum()), 1)
    rec = tp / int(y_dev.sum())
    f1 = 2 * prec * rec / max(prec + rec, 1e-9)
    rows.append((round(th, 2), round(prec, 2), round(rec, 2), round(f1, 2)))
pd.DataFrame(rows, columns=["threshold", "precision", "recall", "f1"])

,threshold,precision,recall,f1
0,0.20,0.52,0.99,0.68
1,0.25,0.58,0.98,0.73
2,0.30,0.64,0.97,0.77
3,0.35,0.70,0.95,0.80
4,0.40,0.75,0.94,0.83
5,0.45,0.79,0.93,0.85
6,0.50,0.80,0.91,0.85
7,0.55,0.82,0.89,0.85
8,0.60,0.84,0.86,0.85
9,0.65,0.85,0.84,0.85


In [6]:
THRESHOLD = 0.35   # change this after reading the table above

def evaluate(data, name):
    p = clf.predict_proba(vec.transform(data["text"]))[:, hz]
    pred = np.where(p >= THRESHOLD, "hazard", "non_hazard")
    print(f"\n{name} (threshold {THRESHOLD}):")
    print(classification_report(data["safety_label"], pred))

evaluate(test_in, "SAME EVENTS")
evaluate(test_out, "UNSEEN EVENTS (floods)")

probs = clf.predict_proba(vec.transform(tests))[:, hz]
for t, p in zip(tests, probs):
    print("HAZARD" if p >= THRESHOLD else "ok    ", f"{p:.2f}", t)


SAME EVENTS (threshold 0.35):
              precision    recall  f1-score   support

      hazard       0.74      0.96      0.84      1001
  non_hazard       0.99      0.91      0.95      3807

    accuracy                           0.92      4808
   macro avg       0.86      0.94      0.89      4808
weighted avg       0.94      0.92      0.92      4808


UNSEEN EVENTS (floods) (threshold 0.35):
              precision    recall  f1-score   support

      hazard       0.46      0.81      0.58        96
  non_hazard       0.98      0.89      0.93       831

    accuracy                           0.88       927
   macro avg       0.72      0.85      0.76       927
weighted avg       0.92      0.88      0.89       927

ok     0.20 water is rushing down 5th avenue, cars are stuck up to the doors
HAZARD 0.48 huge tree fell across the road on Queens Blvd, nobody can pass
ok     0.21 my neighbor is playing loud music again
HAZARD 0.41 basement flooded with sewage after the storm
HAZARD 0.49 

## Findings

- Trained on HumAID tweets (hurricane and flood events, 25,196 tweets after cleaning; 20% hazard = "infrastructure and utility damage").
- Same events: hazard recall 0.92 / precision 0.83. Unseen floods: recall 0.60 / precision 0.69 (small sample, 96 hazard tweets).
- On 7 hand-written sentences the model is no longer confidently wrong, but probabilities stay below 0.5 and ranking is imperfect.
- Tweet-trained model transfers poorly to 311 text (hazard recall 0.17): domain mismatch in style and label definition.
- Design decision: hybrid. Structured 311 reports use category rules; free text uses the text classifier.
- Limits: HumAID labels are noisy and not NYC-specific; bag-of-words does not understand phrases like "street is a river".
- Next: tune the threshold for recall, try a transformer model, then QLoRA Qwen with RAG.